# Занятие 5, пара 1. Особые точки и дескрипторы


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Как узнать тот же участок на другом изображении?

В прошлой теме край говорил: «здесь яркость меняется». Для совмещения двух кадров этого мало:
нужно понять, **какой участок первого кадра соответствует какому участку второго**.
На длинной ровной линии легко сдвинуться вдоль неё и не заметить разницы. Возле уголка такой
сдвиг заметнее. Отсюда начнём, а формулу Харриса соберём после опыта.

**Особая точка** — выбранное место изображения, которое удобно искать повторно.
**Детектор** выбирает места; **дескриптор** описывает окрестность числами, чтобы сравнивать места.
Это разные задачи: «где искать?» и «как узнать?». Угол — один полезный тип особой точки,
но не единственный; отдельный яркий пиксель ещё не делает точку надёжной.

Нужны: знакомые массивы, срезы и градиенты блока 04. Основной код — NumPy и Matplotlib, без OpenCV.
После пары вы сможете объяснить разницу плоского участка, края и угла; построить карту Харриса;
сравнить простые дескрипторы и показать, при каком изменении они ломаются.
Результат — карта точек и небольшой эксперимент с изменением изображения.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### 1. Давайте сначала посмотрим

`np.load` читает готовый массив; `allow_pickle=False` не разрешает загрузку произвольных объектов Python.
Это наша синтетическая карточка, не настоящий документ. `imshow(..., cmap='gray', vmin=0, vmax=1)`
показывает яркости в одной фиксированной шкале. Будем называть точку `(x,y)`, но брать пиксель как `[y,x]`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

image = np.load(DATA_DIR / 'features' / 'card.npy', allow_pickle=False)
plt.figure(figsize=(8,5))
plt.imshow(image, cmap='gray', vmin=0, vmax=1)
plt.scatter([10,25,25], [10,60,20], c=['orange','red','cyan'])
plt.title('Оранжевая: фон; красная: край; голубая: угол')
plt.xlabel('x — столбец'); plt.ylabel('y — строка')
plt.show()


### 2. Сдвинуть окошко и сравнить

Берём окно 7×7 вокруг точки. Сдвигаем его на один пиксель вправо или вниз и сравниваем с исходным.
Из каждой яркости вычитаем яркость на прежнем месте, возводим разности в квадрат и складываем.
Получается **энергия различия**: 0 — окна совпали; большее число — сильнее изменились.
Квадрат не даёт положительным и отрицательным разностям взаимно уничтожиться.

`dx` — сдвиг по x, `dy` — по y. Радиус 3 даёт 7 пикселей, потому что центральный пиксель тоже входит.
Ниже функция делает только этот опыт. Точки выбираем вдали от границы изображения.


In [ ]:
def shift_energy(picture, x, y, dx, dy, radius=3):
    before = picture[y-radius:y+radius+1, x-radius:x+radius+1]
    after = picture[y+dy-radius:y+dy+radius+1, x+dx-radius:x+dx+radius+1]
    difference = after - before
    return np.sum(difference ** 2)

for label, x, y in [('фон',10,10), ('край',25,60), ('угол',25,20)]:
    right = shift_energy(image, x, y, 1, 0)
    down = shift_energy(image, x, y, 0, 1)
    print(label, 'вправо:', round(right,3), 'вниз:', round(down,3))


На фоне оба сдвига почти ничего не меняют. На вертикальном крае заметен сдвиг поперёк,
а вдоль края — нет. У угла меняются оба направления. **Узнаваемость** здесь означает:
маленький сдвиг нельзя незаметно спрятать в каком-нибудь направлении.

Пока мы проверили только два сдвига. Для наклонного края этого недостаточно: изменение может
быть заметно по x и y, но незаметно вдоль диагонали. Харрис учитывает и совместное направление —
поэтому в его формуле ниже появится произведение двух градиентов.


### ✏️ Ваш ход — предсказать сдвиг

Для красной точки(25,60) вычислите `my_right` при сдвиге(1,0)
и `my_down` при(0,1). Для голубой(25,20) — `my_corner_right` и `my_corner_down`.
Используйте `shift_energy`. До запуска нарисуйте стрелку направления, вдоль которого край плохо различим.
В выводе объясните, почему просто «много контраста» недостаточно.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('05.1.1', my_right, my_down, my_corner_right, my_corner_down)


**Мой вывод:** …


### 3. От опыта к карте Харриса

Сдвигать окно вокруг каждого пикселя во всех направлениях дорого. Для малых сдвигов изменение
яркости приближённо описывают градиенты: `Ix` — по горизонтали, `Iy` — по вертикали.
У нас они уже были в теме краёв. Здесь `np.gradient` вычисляет разности соседних яркостей;
возвращает сначала производную по строкам, потом по столбцам, поэтому **Iy, Ix**.
Это простая учебная оценка производной, не оператор Собеля из прошлого блока.


In [ ]:
Iy, Ix = np.gradient(image)
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].imshow(Ix, cmap='coolwarm', vmin=-0.4, vmax=0.4)
axes[0].set_title('Ix: изменение по x')
axes[1].imshow(Iy, cmap='coolwarm', vmin=-0.4, vmax=0.4)
axes[1].set_title('Iy: изменение по y')
plt.show()


Внутри окна соберём три числа:

- `A = сумма(Ix²)` — насколько сильны изменения по x;
- `B = сумма(Ix·Iy)` — как связаны направления изменений;
- `C = сумма(Iy²)` — насколько сильны изменения по y.

Их записывают матрицей `M = [[A,B],[B,C]]`. Она описывает, насколько изменится окно при малом
сдвиге в разных направлениях. Не нужно решать собственные значения, чтобы начать пользоваться методом.
Сначала посчитаем A,B,C только для **одного угла**, окно 7×7.


In [ ]:
x, y = 25, 20
patch_x = Ix[y-3:y+4, x-3:x+4]
patch_y = Iy[y-3:y+4, x-3:x+4]
A = np.sum(patch_x ** 2)
B = np.sum(patch_x * patch_y)
C = np.sum(patch_y ** 2)
M = np.array([[A,B], [B,C]])
print(M)


Отклик Харриса:

$$
R = \det(M)-k\,\mathrm{trace}(M)^2 = AC-B^2-k(A+C)^2.
$$

Определитель `AC−B²` большой, когда в окне есть сильные изменения в двух независимых направлениях.
Вычитание `k(A+C)²` штрафует большой общий контраст, который может оказаться всего лишь краем.
Для опыта возьмём `k=0.04`; это параметр метода, не вероятность и не доля найденных углов.
На идеальном фоне R близок к 0; на сильном крае обычно отрицателен; на угле положителен.
Абсолютная величина зависит от яркости, масштаба и окна — это не универсальная «оценка качества».
[Формула и разбор метода](https://docs.opencv.org/4.x/dc/d0d/tutorial_py_features_harris.html).


In [ ]:
k = 0.04
determinant = A*C - B*B
trace = A+C
corner_response = determinant - k*trace**2
print('Отклик выбранного угла:', corner_response)


### 4. То же для всех пикселей

Нужна сумма значений в окне вокруг каждого пикселя. Это уже знакомая свёртка с ядром из единиц;
для симметричного ядра переворот ничего не меняет. Ниже складываем 49 смещённых участков массива.
`np.pad(..., mode='reflect')` дополняет края отражением; `zeros_like` готовит карту такого же размера.
Функция возвращает **сумму**, не среднее. Для первого просмотра можно читать её как уже разобранный
фильтр окна; ключевой смысл Харриса — какие три поля мы суммируем и зачем.


In [ ]:
def window_sum(values, radius=3):
    height, width = values.shape
    padded = np.pad(values, radius, mode='reflect')
    total = np.zeros_like(values)
    for dy in range(2*radius+1):
        for dx in range(2*radius+1):
            total += padded[dy:dy+height, dx:dx+width]
    return total

A_map = window_sum(Ix ** 2)
B_map = window_sum(Ix * Iy)
C_map = window_sum(Iy ** 2)
R = A_map*C_map - B_map**2 - k*(A_map+C_map)**2


Сначала покажем только положительную часть: `np.maximum(R,0)` заменяет отрицательное нулём
**для картинки**, исходную R не меняет. Светлые пятна — кандидаты в углы. Обратите внимание:
один настоящий угол часто даёт несколько соседних ярких пикселей.


In [ ]:
plt.figure(figsize=(8,4))
plt.imshow(np.maximum(R,0), cmap='magma')
plt.colorbar(label='Положительный отклик R')
plt.title('Карта отклика, ещё не список отдельных точек')
plt.show()


Порог `0.1 * R.max()` отсекает слабый отклик. Но он не убирает несколько соседних точек
одного угла. После порога оставим локальные максимумы: отклик должен быть не меньше соседних
в окне 7×7. На ровном плато может остаться несколько равных максимумов — это честное ограничение
нашего простого отбора; полноценный детектор дополнительно разрежает такие точки.


In [ ]:
threshold = 0.1 * R.max()
local_peak = np.ones_like(R, dtype=bool)
padded_R = np.pad(R, 3, mode='edge')
height, width = R.shape
for dy in range(7):
    for dx in range(7):
        neighbor = padded_R[dy:dy+height, dx:dx+width]
        local_peak &= R >= neighbor
corner_mask = (R > threshold) & local_peak
corner_y, corner_x = np.nonzero(corner_mask)
print('Кандидатов после порога:', np.count_nonzero(R > threshold))
print('Локальных максимумов:', len(corner_x))


`np.nonzero` возвращает индексы истинных элементов маски: сначала строки y, потом столбцы x.
Наложим точки на исходник. Больше точек не обязательно лучше: повторения и нестабильный шум
могут затруднить сопоставление.


In [ ]:
plt.figure(figsize=(8,4))
plt.imshow(image, cmap='gray', vmin=0, vmax=1)
plt.scatter(corner_x, corner_y, s=35, facecolors='none', edgecolors='red')
plt.title('Найденные кандидаты в особые точки')
plt.show()


### ✏️ Ваш ход — прочитать формулу по смыслу

Даны три учебные матрицы M: фон `[[0,0],[0,0]]`,
край `[[8,0],[0,0]]`, угол `[[8,0],[0,8]]`. Для каждой вычислите R по уже разобранной формуле
с k=0.04. Запишите результаты в `my_responses` в этом порядке. Можно использовать цикл.
Какой знак у каждого отклика и почему один сильный градиент не делает угол?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('05.1.2', my_responses)


**Мой вывод:** …


### 5. Нашли место. Как его описать?

Координаты сами по себе бесполезны для узнавания: карточка сдвинулась, координаты тоже.
Возьмём небольшую окрестность и разложим её яркости в один ряд. Такой список чисел — простейший
**дескриптор**. `ravel()` превращает массив в последовательность по строкам.
Сначала окно 2×2, чтобы все числа помещались перед глазами.


In [ ]:
patch = np.array([[0.1,0.8], [0.2,0.9]])
descriptor = patch.ravel()
print(descriptor)
same_patch_brighter = patch + 0.05
other_descriptor = same_patch_brighter.ravel()
raw_difference = descriptor - other_descriptor
raw_distance = np.sqrt(np.sum(raw_difference ** 2))
print('Расстояние после изменения яркости:', raw_distance)


Это евклидово расстояние: корень из суммы квадратов разностей. 0 означает одинаковые числа,
малое значение — похожие числа. Пока даже добавление одинаковой яркости даёт ненулевое расстояние.

Уберём из каждого дескриптора его среднее. Сравнивать будем рисунок **относительно собственной
средней яркости**. Это помогает при одинаковом добавлении яркости без обрезки значений;
не даёт устойчивости ко всем теням, изменению контраста, масштаба или повороту.


In [ ]:
centered = descriptor - descriptor.mean()
other_centered = other_descriptor - other_descriptor.mean()
distance = np.sqrt(np.sum((centered - other_centered) ** 2))
print('После вычитания среднего:', distance)


Вернёмся к карточке. Окно 9×9 вокруг угла превращается в 81 число; операция та же.
`np.linalg.norm` для вектора считает уже знакомый корень из суммы квадратов; используем
его для расстояния между двумя дескрипторами.
`np.rot90` поворачивает массив на 90° против часовой стрелки. Сначала прогноз:
станет ли дескриптор после поворота совпадать сам с собой? Порядок пикселей в ряду изменился.


In [ ]:
corner_patch = image[16:25,21:30]
patch_descriptor = corner_patch.ravel()
patch_descriptor = patch_descriptor - patch_descriptor.mean()
rotated_patch = np.rot90(corner_patch)
rotated_descriptor = rotated_patch.ravel()
rotated_descriptor = rotated_descriptor - rotated_descriptor.mean()
print('Расстояние после поворота:', np.linalg.norm(patch_descriptor - rotated_descriptor))
fig, axes = plt.subplots(1,2,figsize=(6,3))
axes[0].imshow(corner_patch, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Исходный угол')
axes[1].imshow(rotated_patch, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Тот же угол после поворота')
plt.show()


`np.linalg.norm` для одномерного вектора по умолчанию считает то же евклидово расстояние,
которое выше записано через квадрат, сумму и корень. Геометрически угол остался углом,
но наш дескриптор изменился. **Хорошо находить точки и устойчиво описывать их — не одно и то же.**


### ✏️ Ваш ход — проверить обещанную устойчивость

Возьмите `test_patch = np.array([[0.,1.,0.],[0.,0.,0.],[0.,0.,0.]])`.
Сравните после вычитания среднего: исходник и `test_patch+0.2` (результат `my_brightness_distance`),
исходник и `np.rot90(test_patch)` (результат `my_rotation_distance`).
Используйте ravel, mean и norm, уже показанные выше. Запишите, какое изменение наш дескриптор
выдержал и какого свойства у него нет. Мы не обрезаем значения в этом численном опыте.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('05.1.3', my_brightness_distance, my_rotation_distance)


**Мой вывод:** …


### 6. Зачем тогда ORB и SIFT?

Наш дескриптор специально простой, чтобы увидеть его слабость. Практические методы выбирают
масштаб и направление окрестности и строят более устойчивое описание. Это не обещание узнавания
при любом ракурсе, размытии или перекрытии.

ORB использует двоичные сравнения яркости: один вопрос даёт 0 или 1. Такой дескриптор удобно
сравнивать **расстоянием Хэмминга** — числом несовпавших битов. Для вещественных дескрипторов
вроде SIFT обычно используют евклидово расстояние. Тип расстояния выбирают по устройству описания,
а не по тому, какое название звучит солиднее.


In [ ]:
bits_a = np.array([1,0,1,1,0,0,1,0], dtype=np.uint8)
bits_b = np.array([1,0,0,1,1,0,1,0], dtype=np.uint8)
hamming_distance = np.count_nonzero(bits_a != bits_b)
print('Не совпали битов:', hamming_distance)


В этом примере каждый элемент — **один бит**. Настоящий ORB упаковывает 8 бит в байт;
поэтому сравнение числа несовпавших байтов — не расстояние Хэмминга по битам.
Библиотечный matcher умеет считать его правильно.

### Итог

Детектор выбрал место, дескриптор описал окрестность. Теперь можно искать ближайшее описание
на другом изображении — это тема второй пары. Но одинаковые окна могут встретиться в разных местах:
«похож» пока не означает «тот самый».

Сохраните тетрадь с картой точек и выводом опыта. **Запас на 10 минут:** на копии изображения
измените контраст, затем добавьте небольшой шум с фиксированным seed. Повторите расчёт карты;
сравните расположение точек, а не только их количество. Исходный набор не перезаписывайте.

По желанию: [официальный разбор ORB](https://docs.opencv.org/4.x/d1/d89/tutorial_py_orb.html)
и [сопоставление дескрипторов](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html).
В старых статьях сведения о патентах и доступности методов могут устареть; сегодня нам важны
устройство описания и его ограничения, а не запоминание исторических условий.


### Дополнительно при доступном OpenCV: библиотечный детектор

Эта ячейка не нужна основному маршруту. ORB_create создаёт детектор/описатель;
detectAndCompute возвращает точки и их дескрипторы. Для ORB нужна яркость uint8 в 0…255.
Параметры ниже уменьшены под маленькую учебную карточку: это не универсальная настройка.
Если точек нет, выводим это прямо; не считаем отсутствие дескрипторов ошибкой студента.


In [ ]:
try:
    import cv2
except (ImportError, OSError):
    cv2 = None

if cv2 is None:
    print('OpenCV недоступен. Основной опыт уже выполнен на NumPy; сравнение пропускаем.')
else:
    image_u8 = (image * 255).astype(np.uint8)
    orb = cv2.ORB_create(nfeatures=100, edgeThreshold=10, patchSize=15)
    keypoints, descriptors = orb.detectAndCompute(image_u8, None)
    print('Точек ORB:', len(keypoints))
    if descriptors is not None:
        print('Форма дескрипторов:', descriptors.shape, 'тип:', descriptors.dtype)
